# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane, Refresh / Content Opportunity Scoring, is a scoring/ranking task. Pages are not sorted into fixed classes — instead each page gets a priority score, and pages are ranked by that score. High scores mean pages that are still visible/valuable but showing decline or missed opportunity.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The target is a proxy label, not present in the raw data — is_declining_label, derived from the trend_direction column (down = 1). It comes from an observed signal (real measured trend), not an arbitrary invented rule. Later phases will add more proxy targets (CTR opportunity, engagement opportunity) and combine them with weighted business rules into one priority score — but the core ML model here predicts decline probability.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

Precision@K. Since content teams can only review a limited number of pages, the goal is that the top-K ranked pages are truly the highest-opportunity ones. Precision@K tells me, of the top K recommended, how many match high scores on my proxy — a number I can defend to stakeholders since it maps directly to "did we pick well."

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/musab855/flyrank-ml-internship.git"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# One row = one content page, at its current snapshot
lane_cols = [
    "content_id", "client_id", "content_type", "content_age_days",
    "days_since_last_update", "word_count", "impressions_90d",
    "clicks_90d", "sessions_90d", "ctr", "avg_position",
    "engagement_rate", "scroll_rate", "trend_direction", "trend_pct"
]

lane_df = df[lane_cols]

print("Rows:", len(lane_df))
print("Unique content_id:", lane_df["content_id"].nunique())
print("One row = one page:", len(lane_df) == lane_df["content_id"].nunique())

lane_df.head()

Rows: 30000
Unique content_id: 30000
One row = one page: True


,content_id,client_id,content_type,content_age_days,days_since_last_update,word_count,impressions_90d,clicks_90d,sessions_90d,ctr,avg_position,engagement_rate,scroll_rate,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,keyword article,187,20,3221.0,3803,29,17,0.76,10.6,5.88,4.55,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,keyword article,445,25,2481.0,15320,7,9,0.05,20.3,0.00,10.00,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,141,20,3515.0,12581,11,11,0.09,36.5,0.00,28.57,down,-60.9
3,content_331d6c4de07b,client_19581e27de,keyword article,463,22,NaN,11751,58,78,0.49,6.2,1.28,3.45,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,263,14,2803.0,19140,24,145,0.13,44.0,0.00,24.29,down,-34.7


One row = one content page (identified by content_id), with its current performance and freshness signals as of this data snapshot.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In the Week 2 experiment, the hand rule beat the model at Precision@20 (0.90 vs 0.55) but lost at Precision@50 (0.68 vs 0.60–0.72 depending on depth). So the rule isn't "bad" — it's strong at the very top, but runs out of signal deeper in the list. A fixed if-statement can't weigh six+ continuous features together; a model finds where combinations of moderate signals (e.g. older content + mid-range impressions) still predict decline, which a human threshold misses.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.